In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import os

In [ ]:
print("Loading data...")
model_data = pd.read_parquet("../data/processed/preprocessed_data.parquet")

model_data["time_step"] = (model_data["date"] - model_data["date"].min()).dt.days

# Split Date (Last 28 days for validation)
max_date = model_data["date"].max()
split_date = max_date - pd.Timedelta(days=28)

# Separate Train and Validation
train_mask = model_data["date"] < split_date
val_mask = model_data["date"] >= split_date

train_data = model_data[train_mask].copy()
val_data = model_data[val_mask].copy()

# Features for LightGBM (dropping absolute time markers)
drop_cols = ["id", "d", "date", "sales", "wm_yr_wk", "time_step"]
features = [col for col in model_data.columns if col not in drop_cols]



In [ ]:
print("Training Linear Regression (Trend Baseline)...")
lr_model = LinearRegression()

# Train LR using ONLY the time_step to predict sales
X_train_time = train_data[["time_step"]]
y_train_actual = train_data["sales"]

lr_model.fit(X_train_time, y_train_actual)

# Predict the baseline trend for both Train and Validation
train_data["lr_trend_pred"] = lr_model.predict(X_train_time)
val_data["lr_trend_pred"] = lr_model.predict(val_data[["time_step"]])